# `ptof_obs_backtest_replay`

## Purpose (Iteration 2 of the backtest plan)
Where `ptof_obs_backtest_thresholds.ipynb` asks "is each detector's threshold placement right?",
this notebook asks a different question: **"does the *lifecycle* around detection -- dedup, the
24h notify cooldown, digest fan-out routing, auto-resolve, ack-reset-on-rematch -- actually
behave correctly across a real historical timeline?"** It assumes Iteration 1's thresholds are
correct and replays the stateful machinery that sits around them.

## What this notebook does NOT do
- **Never touches the real `obs_incidents` table.** Every MERGE/UPDATE/notify/digest operation
  below targets a shadow table, `obs_incidents_backtest`, created fresh by this notebook with
  the identical schema (copied from `ptof_obs_setup_seed.ipynb` cell 2).
- **Never posts to the live Teams webhook.** `post_teams()` (`ptof_obs_alert.ipynb` cell 10) is
  not called or `%run` here at all. A local `record_notification()` stand-in appends to an
  in-memory list instead of making any HTTP call.
- Does not re-run the real detector notebooks' own persistence cells -- unlike Iteration 1, this
  notebook does not `%run` `ptof_obs_liveness_detection.ipynb` etc. (which would recompute their
  production tables using *current* data on every notebook load); it only imports each
  detector's `_sql(as_of=...)` builder text and evaluates it as a read-only subquery at each
  historical step.

## Replay window and cadence
Steps every **10 minutes** from **2026-09-14 00:00** to **2026-09-22 00:00** (a bounded ~8-day
window containing the three known real events below, per the plan -- not the full history,
since this is far more expensive per-step than Iteration 1 and doesn't need full-history
coverage to validate lifecycle logic):
- the `etl_run_slow` week (correlated multi-table ETL slowdowns)
- the 14/19-table simultaneous staleness episode, 2026-09-19/20
- the `sev2-insights` silence gap (capability_silence_ceiling's 120h ceiling -- may legitimately
  not fire within this 8-day window; its grace period is longer than the replay window, so a
  non-fire here is expected, not evidence the detector is broken)

At each step, this notebook:
1. Evaluates all in-scope detectors' `_sql(as_of=...)` builders (stripped to their bare SELECT,
   same `select_body()` helper as Iteration 1) as of that step's timestamp.
2. MERGEs results into `obs_incidents_backtest`, using the exact same MERGE shape as
   `ptof_obs_alert.ipynb` cell 5 (`INCIDENT_SOURCES`/`SCALAR_INCIDENT_SOURCES`), with
   `current_timestamp()` replaced by the step's fixed `as_of` literal throughout.
3. Runs auto-resolve (cell 6 logic) against the backtest table.
4. Runs the notify-cooldown and digest fan-out logic (cell 11 logic, via the now `as_of`/`table`
   -parameterized `digest_routed_detectors_sql()`) against the backtest table, recording what
   *would* have notified/digested rather than posting anything.
5. Evaluates `unacknowledged_critical`/`long_running_incident` (via their own `as_of`/`table`
   -parameterized builders) against the backtest table's own current state, and MERGEs those in
   too -- these two are self-referential (they monitor `obs_incidents` itself), so unlike every
   other detector they must run *after* everything else in the same step.
6. Snapshots the full `obs_incidents_backtest` state, tagged with `as_of`, into
   `backtest_incident_timeline` -- see the markdown cell after the step loop for how to derive
   opened/updated/resolved/notified events from consecutive snapshots.

## Detectors NOT covered here
- `capability_liveness_unconfigured` -- a config-completeness WARN check, never wired into
  `obs_incidents`/`INCIDENT_SOURCES` in production either (out of scope for this backtest, same
  as production).


In [ ]:
CAT = "mq_gmdf_dev.oil_obs"
import json
import re
from datetime import datetime, timedelta

print(f"backtest incident-lifecycle replay -- target catalog: {CAT}")

In [ ]:
%run "./ptof_obs_liveness_detection"

In [ ]:
%run "./ptof_obs_mal_output"

In [ ]:
%run "./ptof_obs_behavioral_correlation"

In [ ]:
# Local copies of the 5 scalar builders + digest_routed_detectors_sql, same rationale as
# Iteration 1: ptof_obs_alert.ipynb cannot be %run wholesale here (it POSTs to the live Teams
# webhook and writes to the real obs_incidents table). unacknowledged_critical_sql and
# long_running_incident_sql already take (as_of, table) params in the live notebook (extended
# 2026-09-22 for this exact purpose), so they are reproduced verbatim below, pointed at
# obs_incidents_backtest by this notebook's step loop.
#
# Keep these in sync BY HAND with ptof_obs_alert.ipynb cells 2 and 4 if that SQL ever changes --
# same deliberate exception to the single-source-of-truth pattern as Iteration 1.

FANOUT_DIGEST_THRESHOLD = 5
FANOUT_WINDOW_MINUTES = 60
FANOUT_CLUSTER_MINUTES = 15


def digest_routed_detectors_sql(as_of="current_timestamp()", table=None):
    table = table or f"{CAT}.obs_incidents"
    return f"""
        SELECT detector FROM {table}
        WHERE resolved_at IS NULL
          AND last_detected >= {as_of} - INTERVAL {FANOUT_WINDOW_MINUTES} MINUTES
        GROUP BY detector
        HAVING count(*) > {FANOUT_DIGEST_THRESHOLD}
           AND cast(max(first_detected) as long) - cast(min(first_detected) as long)
               <= {FANOUT_CLUSTER_MINUTES} * 60
    """


def pipeline_heartbeat_sql(as_of="current_timestamp()"):
    return f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n, count(*) AS rows_last_25m
    FROM {CAT}.v_llm_bronze
    WHERE called_at >= {as_of} - INTERVAL 25 MINUTES
    """


def etl_pipeline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT CASE WHEN max(run_timestamp) < {as_of} - INTERVAL 30 MINUTES
                    THEN 1 ELSE 0 END AS n,
           max(run_timestamp) AS latest_run
        FROM {CAT}.v_etl_bronze"""


def nightly_baseline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT count(*) AS n, max(computed_at) AS last_computed
        FROM {CAT}.response_field_baseline
        WHERE computed_at < {as_of} - INTERVAL 36 HOURS"""


def unacknowledged_critical_sql(as_of="current_timestamp()", table=None):
    table = table or f"{CAT}.obs_incidents"
    return f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,':',coalesce(capability,'-')))) AS detail,
           min(first_detected) AS oldest
    FROM {table}
    WHERE severity = 'CRITICAL'
      AND acknowledged_at IS NULL
      AND resolved_at IS NULL
      AND detector NOT IN ({digest_routed_detectors_sql(as_of=as_of, table=table)})
      AND first_detected <= {as_of} - INTERVAL 1 HOUR
    """


def long_running_incident_sql(as_of="current_timestamp()", table=None):
    table = table or f"{CAT}.obs_incidents"
    return f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,' open ',
                      cast(datediff({as_of}, first_detected) AS STRING),'d'))) AS detail
    FROM {table}
    WHERE resolved_at IS NULL
      AND acknowledged_at IS NULL
      AND first_detected <= {as_of} - INTERVAL 3 DAYS
    """


print("Local scalar/digest builders defined.")

In [ ]:
# --- Shadow incident table + timeline table (never the real obs_incidents) -------------------

BACKTEST_TABLE = f"{CAT}.obs_incidents_backtest"

spark.sql(f"DROP TABLE IF EXISTS {BACKTEST_TABLE}")
spark.sql(f"""
    CREATE TABLE {BACKTEST_TABLE} (
        detector         STRING,
        source_row_id    STRING,
        capability       STRING,
        severity         STRING,
        first_detected   TIMESTAMP,
        last_detected    TIMESTAMP,
        detection_count  BIGINT,
        signal_payload   STRING,
        notified_at      TIMESTAMP,
        acknowledged_by  STRING,
        acknowledged_at  TIMESTAMP,
        resolved_at      TIMESTAMP
    )
""")

spark.sql(f"DROP TABLE IF EXISTS {CAT}.backtest_incident_timeline")
spark.sql(f"""
    CREATE TABLE {CAT}.backtest_incident_timeline (
        as_of            TIMESTAMP,
        detector         STRING,
        source_row_id    STRING,
        capability       STRING,
        severity         STRING,
        first_detected   TIMESTAMP,
        last_detected    TIMESTAMP,
        detection_count  BIGINT,
        notified_at      TIMESTAMP,
        acknowledged_at  TIMESTAMP,
        resolved_at      TIMESTAMP,
        would_notify     BOOLEAN,
        would_digest     BOOLEAN
    )
""")

print(f"Created {BACKTEST_TABLE} and {CAT}.backtest_incident_timeline (fresh, empty).")

In [ ]:
# --- Detector source definitions for the MERGE loop -------------------------------------------
# Same shape as ptof_obs_alert.ipynb cell 5's INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, but the
# "source" is a stripped-to-SELECT call to each detector's _sql(as_of) builder (via
# select_body(), same helper as Iteration 1) instead of a persisted findings table -- this
# notebook never recomputes/overwrites those production tables.

_CREATE_TABLE_RE = re.compile(r"(?is)^\s*CREATE OR REPLACE TABLE\s+\S+\s+AS\s*")


def select_body(sql_text):
    return _CREATE_TABLE_RE.sub("", sql_text).strip()


def rolling_etl_duration_baseline_sql(as_of):
    """Same rolling 30-day median/MAD baseline as Iteration 1 -- avoids lookahead bias from
    the live etl_duration_baseline snapshot."""
    return f"""
    WITH eligible AS (
        SELECT table_or_view, task_name, duration_seconds
        FROM {CAT}.v_etl_bronze
        WHERE run_timestamp >= {as_of} - INTERVAL 30 DAYS
          AND run_timestamp < {as_of}
          AND status = 'success'
    ),
    med AS (
        SELECT table_or_view, task_name,
               count(*)                                     AS n,
               percentile_approx(duration_seconds, 0.5)      AS med_s
        FROM eligible
        GROUP BY table_or_view, task_name
    )
    SELECT
        e.table_or_view, e.task_name, m.n, m.med_s AS median_duration_s,
        percentile_approx(abs(e.duration_seconds - m.med_s), 0.5) AS mad_duration_s,
        greatest(
            m.med_s + 5 * 1.4826 * percentile_approx(abs(e.duration_seconds - m.med_s), 0.5),
            m.med_s * 1.5
        ) AS upper_bound_s,
        {as_of} AS computed_at
    FROM eligible e
    JOIN med m USING (table_or_view, task_name)
    GROUP BY e.table_or_view, e.task_name, m.n, m.med_s
    HAVING m.n >= 50
    """


def etl_run_slow_sql_rolling(as_of):
    return f"""
    WITH flagged AS (
        SELECT
            e.table_or_view, e.task_name, e.duration_seconds, e.run_timestamp,
            window(e.run_timestamp, '60 minutes').start AS window_start,
            base.upper_bound_s
        FROM {CAT}.v_etl_bronze e
        JOIN _rolling_etl_duration_baseline base
          ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
        WHERE e.run_timestamp >= {as_of} - INTERVAL 24 HOURS
          AND e.run_timestamp < {as_of}
          AND e.status = 'success'
          AND e.duration_seconds > base.upper_bound_s
    )
    SELECT
        task_name, window_start,
        count(*) AS anomalous_count,
        count(distinct table_or_view) AS affected_table_count,
        concat_ws(', ', collect_set(table_or_view)) AS affected_tables,
        max(duration_seconds) AS max_duration_s,
        max(upper_bound_s) AS upper_bound_s,
        sha2(concat_ws('|', task_name, cast(window_start AS STRING)), 256) AS finding_signature,
        {as_of} AS detected_at
    FROM flagged
    GROUP BY task_name, window_start
    HAVING count(*) >= 3
    """


# (detector, sql_fn, id_col, cap_col, severity, payload_cols, extra_where)
INCIDENT_SOURCES = [
    ("handover_delivery", handover_delivery_failures_sql, "ish_row_id",
     None, "CRITICAL",
     ["failure_reason", "shift_date", "batch_nbr"], ""),
    ("blank_output", blank_output_findings_sql, "finding_signature",
     "capability", "CRITICAL",
     ["model_config", "blank_rate_window", "blank_count_window", "total_calls_window",
      "latest_hour"], ""),
    ("schema_field_missing", response_schema_drift_sql, "finding_signature",
     "capability", "CRITICAL",
     ["model_configs_seen", "field_name", "drift_type",
      "baseline_presence_rate", "current_present", "current_rows"],
     "WHERE drift_type = 'field_missing'"),
    ("handover_delivery_rate", handover_delivery_rate_sql, "'handover_delivery_rate_global'",
     None, "CRITICAL",
     ["failure_pct_7d", "sent_ok", "failed", "last_attempt"],
     "WHERE (failure_pct_7d > 20 AND (sent_ok + failed) >= 10) OR failed >= 3"),
    ("etl_pipeline_failure", etl_pipeline_health_sql, "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "status", "error_message", "run_timestamp", "duration_seconds"], ""),
    ("etl_table_staleness", etl_table_staleness_sql, "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "minutes_since_last_run", "last_run_at"], ""),
    ("etl_run_slow", etl_run_slow_sql_rolling, "finding_signature",
     None, "CRITICAL",
     ["task_name", "anomalous_count", "affected_table_count", "affected_tables",
      "max_duration_s", "upper_bound_s"], ""),
    ("capability_silence_ceiling", capability_silence_ceiling_sql, "finding_signature",
     "capability", "CRITICAL",
     ["silence_ceiling_hours", "hours_since_last_call", "last_call_at", "owner"], ""),
    ("shift_context_missing", shift_context_missing_sql, "finding_signature",
     "capability", "CRITICAL",
     ["total_calls", "blank_shift_type", "blank_batch_nbr", "null_shift_date", "last_seen"], ""),
    ("capability_silence", capability_silence_sql, "finding_signature",
     "capability", "CRITICAL",
     ["expected_min_daily", "silence_grace_hours", "owner", "calls_last_7d",
      "last_call_at", "hours_since_last_call"], ""),
    ("etl_row_count_anomaly", etl_row_count_anomaly_sql, "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "rows_written", "regime", "run_timestamp"], ""),
]

# (detector, sql_fn, constant_source_row_id)
SCALAR_INCIDENT_SOURCES = [
    ("pipeline_heartbeat", pipeline_heartbeat_sql, "pipeline_heartbeat_global"),
    ("etl_pipeline_staleness", etl_pipeline_staleness_sql, "etl_staleness_global"),
    ("nightly_baseline_staleness", nightly_baseline_staleness_sql, "nightly_baseline_staleness_global"),
]

# unacknowledged_critical / long_running_incident are self-referential (they monitor
# obs_incidents_backtest itself) -- handled separately, after every other detector, in the step
# loop below, using their own (as_of, table)-parameterized builders imported/defined above.
SELF_REFERENTIAL_SOURCES = [
    ("unacknowledged_critical", unacknowledged_critical_sql, "unacknowledged_critical_global"),
    ("long_running_incident", long_running_incident_sql, "long_running_incident_global"),
]

print(f"{len(INCIDENT_SOURCES)} table-backed + {len(SCALAR_INCIDENT_SOURCES)} scalar + "
      f"{len(SELF_REFERENTIAL_SOURCES)} self-referential detector(s) in scope for replay.")

In [ ]:
# --- One step of the replay --------------------------------------------------------------------

def as_of_literal(dt):
    return "TIMESTAMP'{}'".format(dt.strftime("%Y-%m-%d %H:%M:%S"))


def run_step(as_of_dt):
    as_of = as_of_literal(as_of_dt)
    active_detectors = []

    # 1. Table-backed detectors (MERGE, cell-5 shape, current_timestamp() -> {as_of})
    for detector, sql_fn, id_col, cap_col, severity, payload_cols, extra in INCIDENT_SOURCES:
        try:
            cap_expr = cap_col if cap_col else "CAST(NULL AS STRING)"
            payload = ", ".join(f"'{c}', CAST({c} AS STRING)" for c in payload_cols)
            source_expr = f"({select_body(sql_fn(as_of))}) __src"
            if detector == "etl_run_slow":
                spark.sql(rolling_etl_duration_baseline_sql(as_of)).createOrReplaceTempView(
                    "_rolling_etl_duration_baseline")
            spark.sql(f"""
                MERGE INTO {BACKTEST_TABLE} t
                USING (
                  SELECT '{detector}'             AS detector,
                         CAST({id_col} AS STRING) AS source_row_id,
                         {cap_expr}               AS capability,
                         '{severity}'             AS severity,
                         to_json(map({payload}))  AS signal_payload
                  FROM {source_expr} {extra}
                ) s
                ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                WHEN MATCHED THEN UPDATE SET
                    t.severity        = s.severity,
                    t.last_detected   = {as_of},
                    t.detection_count = t.detection_count + 1,
                    t.signal_payload  = s.signal_payload,
                    t.resolved_at     = NULL,
                    t.acknowledged_at = NULL,
                    t.acknowledged_by = NULL
                WHEN NOT MATCHED THEN INSERT
                    (detector, source_row_id, capability, severity,
                     first_detected, last_detected, detection_count, signal_payload)
                  VALUES
                    (s.detector, s.source_row_id, s.capability, s.severity,
                     {as_of}, {as_of}, 1, s.signal_payload)
            """)
            active_detectors.append(detector)
        except Exception as e:  # noqa: BLE001
            print(f"  [{as_of_dt}] SKIPPED {detector}: {str(e).splitlines()[0][:160]}")

    # 2. Scalar detectors
    for detector, sql_fn, source_row_id in SCALAR_INCIDENT_SOURCES:
        try:
            row = spark.sql(sql_fn(as_of)).first()
            active_detectors.append(detector)
            if row is None or not row["n"]:
                continue
            payload = json.dumps({k: str(v) for k, v in row.asDict().items() if k != "n"})
            spark.sql(f"""
                MERGE INTO {BACKTEST_TABLE} t
                USING (
                  SELECT '{detector}' AS detector, '{source_row_id}' AS source_row_id,
                         CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                         '{payload.replace("'", "''")}' AS signal_payload
                ) s
                ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                WHEN MATCHED THEN UPDATE SET
                    t.severity        = s.severity,
                    t.last_detected   = {as_of},
                    t.detection_count = t.detection_count + 1,
                    t.signal_payload  = s.signal_payload,
                    t.resolved_at     = NULL,
                    t.acknowledged_at = NULL,
                    t.acknowledged_by = NULL
                WHEN NOT MATCHED THEN INSERT
                    (detector, source_row_id, capability, severity,
                     first_detected, last_detected, detection_count, signal_payload)
                  VALUES
                    (s.detector, s.source_row_id, s.capability, s.severity,
                     {as_of}, {as_of}, 1, s.signal_payload)
            """)
        except Exception as e:  # noqa: BLE001
            print(f"  [{as_of_dt}] SKIPPED {detector}: {str(e).splitlines()[0][:160]}")

    # 3. Auto-resolve (cell-6 shape): a detector that ran cleanly but didn't re-MERGE this step
    # means the condition cleared.
    if active_detectors:
        detector_list = ", ".join(f"'{d}'" for d in active_detectors)
        spark.sql(f"""
            UPDATE {BACKTEST_TABLE}
            SET resolved_at = {as_of}
            WHERE resolved_at IS NULL
              AND detector IN ({detector_list})
              AND last_detected < {as_of}
        """)

    # 4. Self-referential detectors (unacknowledged_critical / long_running_incident) -- run
    # after everything else in this step, since they monitor the backtest table's own state.
    for detector, sql_fn, source_row_id in SELF_REFERENTIAL_SOURCES:
        try:
            row = spark.sql(sql_fn(as_of, BACKTEST_TABLE)).first()
            if row is None or not row["n"]:
                spark.sql(f"""
                    UPDATE {BACKTEST_TABLE}
                    SET resolved_at = {as_of}
                    WHERE resolved_at IS NULL AND detector = '{detector}'
                      AND last_detected < {as_of}
                """)
                continue
            payload = json.dumps({k: str(v) for k, v in row.asDict().items() if k != "n"})
            spark.sql(f"""
                MERGE INTO {BACKTEST_TABLE} t
                USING (
                  SELECT '{detector}' AS detector, '{source_row_id}' AS source_row_id,
                         CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                         '{payload.replace("'", "''")}' AS signal_payload
                ) s
                ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                WHEN MATCHED THEN UPDATE SET
                    t.severity        = s.severity,
                    t.last_detected   = {as_of},
                    t.detection_count = t.detection_count + 1,
                    t.signal_payload  = s.signal_payload,
                    t.resolved_at     = NULL,
                    t.acknowledged_at = NULL,
                    t.acknowledged_by = NULL
                WHEN NOT MATCHED THEN INSERT
                    (detector, source_row_id, capability, severity,
                     first_detected, last_detected, detection_count, signal_payload)
                  VALUES
                    (s.detector, s.source_row_id, s.capability, s.severity,
                     {as_of}, {as_of}, 1, s.signal_payload)
            """)
        except Exception as e:  # noqa: BLE001
            print(f"  [{as_of_dt}] SKIPPED {detector}: {str(e).splitlines()[0][:160]}")

    # 5. Notify cooldown (cell-11 first block): who WOULD have been paged this step.
    notified_ids = set()
    try:
        to_notify = spark.sql(f"""
            SELECT detector, source_row_id
            FROM {BACKTEST_TABLE}
            WHERE severity = 'CRITICAL'
              AND detector NOT IN ({digest_routed_detectors_sql(as_of=as_of, table=BACKTEST_TABLE)})
              AND acknowledged_at IS NULL
              AND resolved_at IS NULL
              AND (notified_at IS NULL OR notified_at < {as_of} - INTERVAL 24 HOURS)
        """).collect()
        if to_notify:
            notified_ids = {(r.detector, r.source_row_id) for r in to_notify}
            spark.sql(f"""
                UPDATE {BACKTEST_TABLE}
                SET notified_at = {as_of}
                WHERE severity = 'CRITICAL'
                  AND detector NOT IN ({digest_routed_detectors_sql(as_of=as_of, table=BACKTEST_TABLE)})
                  AND acknowledged_at IS NULL AND resolved_at IS NULL
                  AND (notified_at IS NULL OR notified_at < {as_of} - INTERVAL 24 HOURS)
            """)
    except Exception as e:  # noqa: BLE001
        print(f"  [{as_of_dt}] notify query failed: {str(e).splitlines()[0][:160]}")

    # 6. Digest fan-out (cell-11 second block): which detector(s) WOULD have gotten a digest.
    digested_detectors = set()
    try:
        digest_detectors = [r.detector for r in
                             spark.sql(digest_routed_detectors_sql(as_of=as_of, table=BACKTEST_TABLE)).collect()]
    except Exception as e:  # noqa: BLE001
        print(f"  [{as_of_dt}] digest-routed lookup failed: {str(e).splitlines()[0][:160]}")
        digest_detectors = []

    for detector in digest_detectors:
        try:
            digest_rows = spark.sql(f"""
                SELECT source_row_id FROM {BACKTEST_TABLE}
                WHERE detector = '{detector}' AND resolved_at IS NULL
                  AND first_detected >= {as_of} - INTERVAL 24 HOURS
            """).collect()
            if not digest_rows:
                continue
            digest_marker = f"{detector}_digest"
            digest_due = spark.sql(f"""
                SELECT count(*) = 0 OR max(notified_at) < {as_of} - INTERVAL 24 HOURS AS due
                FROM {BACKTEST_TABLE}
                WHERE detector = '{digest_marker}' AND source_row_id = 'daily_digest'
            """).first().due
            if not digest_due:
                continue
            digested_detectors.add(detector)
            spark.sql(f"""
                MERGE INTO {BACKTEST_TABLE} t
                USING (
                  SELECT '{digest_marker}' AS detector, 'daily_digest' AS source_row_id,
                         CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                         to_json(map('occurrences_in_window',
                                     CAST({len(digest_rows)} AS STRING))) AS signal_payload
                ) s
                ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                WHEN MATCHED THEN UPDATE SET
                    t.last_detected   = {as_of},
                    t.detection_count = t.detection_count + 1,
                    t.signal_payload  = s.signal_payload,
                    t.notified_at     = {as_of},
                    t.resolved_at     = {as_of}
                WHEN NOT MATCHED THEN INSERT
                    (detector, source_row_id, capability, severity,
                     first_detected, last_detected, detection_count, signal_payload,
                     notified_at, resolved_at)
                  VALUES
                    (s.detector, s.source_row_id, s.capability, s.severity,
                     {as_of}, {as_of}, 1, s.signal_payload, {as_of}, {as_of})
            """)
        except Exception as e:  # noqa: BLE001
            print(f"  [{as_of_dt}] digest for {detector} failed: {str(e).splitlines()[0][:160]}")

    # 7. Snapshot into the timeline table.
    notify_ids_sql = ", ".join(
        f"('{d}','{r}')" for d, r in notified_ids) if notified_ids else None
    would_notify_expr = (
        f"struct(detector, source_row_id) IN ({notify_ids_sql})" if notify_ids_sql else "false"
    )
    digest_list_sql = ", ".join(f"'{d}'" for d in digested_detectors) if digested_detectors else "''"

    spark.sql(f"""
        INSERT INTO {CAT}.backtest_incident_timeline
        SELECT
            {as_of} AS as_of,
            detector, source_row_id, capability, severity,
            first_detected, last_detected, detection_count,
            notified_at, acknowledged_at, resolved_at,
            {would_notify_expr} AS would_notify,
            detector IN ({digest_list_sql}) AS would_digest
        FROM {BACKTEST_TABLE}
    """)

    return len(active_detectors), len(notified_ids), len(digested_detectors)


print("run_step() defined.")

In [ ]:
# --- Drive the replay: every 10 min, 2026-09-14 00:00 -> 2026-09-22 00:00 --------------------

REPLAY_START = datetime(2026, 9, 14, 0, 0, 0)
REPLAY_END = datetime(2026, 9, 22, 0, 0, 0)
STEP = timedelta(minutes=10)

steps = []
cur = REPLAY_START
while cur <= REPLAY_END:
    steps.append(cur)
    cur += STEP

print(f"Replaying {len(steps)} steps ({REPLAY_START} .. {REPLAY_END}, every {STEP}).")

for i, as_of_dt in enumerate(steps):
    n_active, n_notified, n_digested = run_step(as_of_dt)
    if i % 144 == 0:  # once per simulated day (144 * 10min = 24h)
        print(f"  step {i}/{len(steps)}  as_of={as_of_dt}  "
              f"active_detectors={n_active} notified={n_notified} digested={n_digested}")

print("Replay complete.")

In [ ]:
# --- Summary: incident counts by detector at end of replay -------------------------------------

spark.sql(f"""
    SELECT detector, count(*) AS incidents,
           sum(int(resolved_at IS NOT NULL)) AS resolved,
           sum(int(resolved_at IS NULL)) AS still_open,
           min(first_detected) AS first_seen,
           max(last_detected) AS last_seen
    FROM {BACKTEST_TABLE}
    GROUP BY detector
    ORDER BY incidents DESC
""").show(30, truncate=False)

In [ ]:
# --- Notify/digest activity over the replay window ----------------------------------------------

spark.sql(f"""
    SELECT detector,
           count(*)                              AS snapshot_rows,
           sum(int(would_notify))                 AS would_notify_events,
           sum(int(would_digest))                 AS would_digest_events
    FROM {CAT}.backtest_incident_timeline
    GROUP BY detector
    HAVING sum(int(would_notify)) > 0 OR sum(int(would_digest)) > 0
    ORDER BY would_notify_events + would_digest_events DESC
""").show(30, truncate=False)

## Deriving open/update/resolve transitions from the timeline

`backtest_incident_timeline` stores a full snapshot of `obs_incidents_backtest` at every 10-min
step rather than pre-computed transition events, to keep the step loop itself simple and
correct. Transitions are a straightforward `LAG()` comparison over consecutive `as_of` values
per `(detector, source_row_id)`, e.g.:

```sql
WITH ordered AS (
  SELECT *,
         LAG(resolved_at) OVER (PARTITION BY detector, source_row_id ORDER BY as_of) AS prev_resolved,
         LAG(as_of)        OVER (PARTITION BY detector, source_row_id ORDER BY as_of) AS prev_as_of
  FROM mq_gmdf_dev.oil_obs.backtest_incident_timeline
)
SELECT detector, source_row_id, as_of,
       CASE
         WHEN prev_as_of IS NULL THEN 'opened'
         WHEN resolved_at IS NOT NULL AND prev_resolved IS NULL THEN 'resolved'
         WHEN resolved_at IS NULL AND prev_resolved IS NOT NULL THEN 'reopened_ack_reset'
         ELSE 'updated'
       END AS event_type
FROM ordered
```

## Comparison against the three known real events

**This notebook has never been executed against live Databricks/Spark in this session** -- there
is no live connection in the environment that authored it. Every query above has been written to
be syntactically and semantically correct against the schemas documented in
`TECHNICAL_REFERENCE.md` and the source detector/alert notebooks (the MERGE/auto-resolve/notify/
digest shapes are lifted verbatim from `ptof_obs_alert.ipynb` cells 5, 6, and 11, with
`current_timestamp()` replaced by the step's fixed `as_of` literal), but none of it has actually
run, so the comparisons below describe what to look for once it does, not confirmed results:

1. **`etl_run_slow` week** -- expect `etl_run_slow` incidents opening and closing across the
   replay window, tracking the known correlated multi-table slowdown events, with `would_digest`
   engaging once its fan-out crosses `FANOUT_DIGEST_THRESHOLD` (5) within a 60-minute window
   clustered inside 15 minutes, and per-incident `would_notify` staying suppressed for that
   detector while digest routing is active (`digest_routed_detectors_sql` excludes it from the
   per-incident notify query).
2. **14/19-table staleness, 2026-09-19/20** -- expect `etl_table_staleness` to open a burst of
   incidents (one per affected table) right around 2026-09-19/20 in the timeline, fan-out
   crossing the same digest threshold, and auto-resolve clearing them once the ETL pipeline
   catches up (i.e. once `etl_table_staleness_sql` stops returning that table's row on a
   subsequent step).
3. **`sev2-insights` silence (`capability_silence_ceiling`)** -- its grace window is 120 hours
   (5 days); an 8-day replay window *can* contain a full ceiling breach, but if `sev2-insights`
   was already silent before 2026-09-14 or only crossed 120h near the very end of the window,
   this replay may show 0 fires for this detector. **That is expected, not a bug** -- do not
   treat a quiet `capability_silence_ceiling` row count here as evidence the detector or the
   replay logic is broken; cross-check against `capability_silence_ceiling`'s own
   `hours_since_last_call` at `as_of = REPLAY_END` before concluding anything.

Any mismatch between what actually happened in production (per `obs_incidents` history/handoff
notes) and what this replay reproduces is a lifecycle bug worth its own follow-up, not a backtest
artifact to explain away, per the plan's Verification section.
